# Adım 1 — Veri Altyapısı

## IEEE-CIS Fraud Detection

Bu notebook IEEE-CIS Fraud Detection veri setinin veri altyapısını kurar.  
İki veri kaynağı (`train_transaction.csv` ve `train_identity.csv`) yüklenerek birleştirilir,  
veri kalitesi kapsamlı raporlarla incelenir.

**Notebook Yapısı:**
1. Kütüphaneler ve veri yükleme
2. İlk inceleme
3. Veri birleştirme (merge)
4. Eksik veri raporu
5. Kolon tipi tespiti
6. Kardinalite raporu
7. Sabit kolonlar
8. Duplikasyon raporu
9. Geçerlilik raporu
10. Outlier raporu
11. Dağılım raporu
12. Kalite özeti
13. Nadir kombinasyon raporu
14. Kolon ilişki raporu
15. Entity davranış raporu

## 1. Kütüphaneler ve Veri Yükleme

In [1]:
import pandas as pd
import numpy as np

In [2]:
identity_data = pd.read_csv('../train_identity.csv')

trx_data = pd.read_csv('../train_transaction.csv')

## 2. İlk İnceleme

Veri setlerinin boyutları ve örnek satırlar inceleniyor.

| Veri Seti | Satır | Sütun |
|-----------|-------|-------|
| Identity | 144,233 | 41 |
| Transaction | 590,540 | 394 |

In [3]:
identity_data.sample(3)

,TransactionID,id_01,id_02,id_03,id_04,id_05,id_06,id_07,id_08,id_09,...,id_31,id_32,id_33,id_34,id_35,id_36,id_37,id_38,DeviceType,DeviceInfo
138244,3547473,-10.0,1974.0,0.0,0.0,NaN,NaN,NaN,NaN,0.0,...,NaN,24.0,1366x768,match_status:2,T,F,T,F,desktop,Windows
5777,3009679,-100.0,546991.0,NaN,NaN,-8.0,-26.0,NaN,NaN,NaN,...,mobile safari 10.0,NaN,NaN,NaN,F,F,F,F,mobile,NaN
39272,3082850,-5.0,163396.0,NaN,NaN,0.0,0.0,NaN,NaN,NaN,...,chrome 63.0,NaN,NaN,NaN,F,F,T,T,desktop,NaN


In [4]:
trx_data.sample(3)

,TransactionID,isFraud,TransactionDT,TransactionAmt,ProductCD,card1,card2,card3,card4,card5,...,V330,V331,V332,V333,V334,V335,V336,V337,V338,V339
155598,3142598,0,3216164,24.00,W,6592,278.0,150.0,visa,226.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
413661,3400661,0,10441792,87.00,W,10112,360.0,150.0,visa,166.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
249230,3236230,0,5942373,35.95,W,5700,122.0,150.0,mastercard,166.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [5]:
print('Toplam Satır ve Sütun Sayıları: \nIdentiy Data = {0} \nTransaction Data = {1}'.format(identity_data.shape, trx_data.shape))

Toplam Satır ve Sütun Sayıları: 
Identiy Data = (144233, 41) 
Transaction Data = (590540, 394)


### ID Eşleşme Analizi

Identity ve Transaction tabloları `TransactionID` üzerinden join edilecek.  
Önce eşleşme oranı kontrol ediliyor.

> **Bulgu:** 144,233 identity kaydının tamamı transaction tablosunda mevcut (%100 eşleşme).

In [6]:
customer_ids = set(identity_data["TransactionID"])
transaction_ids = set(trx_data["TransactionID"])

# Customer ID'lerin kaç tanesi transaction içinde var
common_ids = customer_ids.intersection(transaction_ids)

print(f"Customer DF toplam ID: {len(customer_ids)}")
print(f"Transaction DF toplam ID: {len(transaction_ids)}")
print(f"Ortak ID sayısı: {len(common_ids)}")

# Hepsi transaction içinde var mı?
all_exist = customer_ids.issubset(transaction_ids)

print("Tüm customer ID'leri transaction içinde var mı?:", all_exist)

# Yüzde hesabı
percentage = (len(common_ids) / len(customer_ids)) * 100

print(f"Eşleşme yüzdesi: %{percentage:.2f}")

Customer DF toplam ID: 144233
Transaction DF toplam ID: 590540
Ortak ID sayısı: 144233
Tüm customer ID'leri transaction içinde var mı?: True
Eşleşme yüzdesi: %100.00


## 3. Yardımcı Fonksiyonlar

Aşağıdaki raporlama fonksiyonları tanımlanıyor. Her fonksiyon işaretleyici prensibiyle çalışır — veri silmez, yalnızca raporlar.

### `missing_report(df)`
Kolon bazında eksik değer sayısı ve oranını döndürür. Yalnızca eksik değeri olan kolonları gösterir.

In [7]:
def missing_report(df):
    report = pd.DataFrame({
        "missing_count": df.isnull().sum(),
        "missing_ratio (%)": (df.isnull().sum() / len(df)) * 100
    })

    # Sadece boş değeri olan kolonları göster
    report = report[report["missing_count"] > 0]

    # Çoktan aza sırala
    report = report.sort_values(by="missing_count", ascending=False)

    return report

### `dataframe_report(df)`
Her kolon için dtype, eksik değer ve istatistikleri (sayısal için mean/min/max, kategorik için unique sayısı) döndürür.

In [8]:
def dataframe_report(df):
    report_rows = []

    for col in df.columns:
        series = df[col]

        dtype = series.dtype
        missing_count = series.isnull().sum()
        missing_ratio = (missing_count / len(df)) * 100

        row = {
            "column": col,
            "dtype": str(dtype),
            "missing_count": missing_count,
            "missing_ratio (%)": round(missing_ratio, 2),
        }

        # Sayısal kolonlar
        if pd.api.types.is_numeric_dtype(series):

            row["mean"] = round(series.mean(), 4) if not series.isnull().all() else np.nan
            row["min"] = series.min()
            row["max"] = series.max()

        # Kategorik / object kolonlar
        else:
            row["unique_values"] = series.nunique(dropna=True)

        report_rows.append(row)

    report = pd.DataFrame(report_rows)

    return report

## 4. DataFrame Raporu

Identity ve Transaction veri setleri için kolon bazlı özet raporlar üretiliyor.

### Identity Veri Seti

In [9]:
report_df_identity = dataframe_report(identity_data)

In [10]:
report_df_identity

,column,dtype,missing_count,missing_ratio (%),mean,min,max,unique_values
0,TransactionID,int64,0,0.00,3.236329e+06,2987004.0,3577534.0,NaN
1,id_01,float64,0,0.00,-1.017050e+01,-100.0,0.0,NaN
2,id_02,float64,3361,2.33,1.747166e+05,1.0,999595.0,NaN
3,id_03,float64,77909,54.02,6.020000e-02,-13.0,10.0,NaN
4,id_04,float64,77909,54.02,-5.890000e-02,-28.0,0.0,NaN
5,id_05,float64,7368,5.11,1.615600e+00,-72.0,52.0,NaN
6,id_06,float64,7368,5.11,-6.698700e+00,-100.0,0.0,NaN
7,id_07,float64,139078,96.43,1.328540e+01,-46.0,61.0,NaN
8,id_08,float64,139078,96.43,-3.860040e+01,-100.0,0.0,NaN
9,id_09,float64,69307,48.05,9.100000e-02,-36.0,25.0,NaN


> Unique değer sayısına göre sıralı görünüm:

In [11]:
report_df_identity.sort_values(by = 'unique_values', ascending=False)

,column,dtype,missing_count,missing_ratio (%),mean,min,max,unique_values
40,DeviceInfo,object,25567,17.73,NaN,NaN,NaN,1786.0
33,id_33,object,70944,49.19,NaN,NaN,NaN,260.0
31,id_31,object,3951,2.74,NaN,NaN,NaN,130.0
30,id_30,object,66668,46.22,NaN,NaN,NaN,75.0
34,id_34,object,66428,46.06,NaN,NaN,NaN,4.0
23,id_23,object,139064,96.42,NaN,NaN,NaN,3.0
15,id_15,object,3248,2.25,NaN,NaN,NaN,3.0
27,id_27,object,139064,96.42,NaN,NaN,NaN,2.0
28,id_28,object,3255,2.26,NaN,NaN,NaN,2.0
29,id_29,object,3255,2.26,NaN,NaN,NaN,2.0


### Transaction Veri Seti

In [12]:
report_df_trx = dataframe_report(trx_data)

In [13]:
report_df_trx

,column,dtype,missing_count,missing_ratio (%),mean,min,max,unique_values
0,TransactionID,int64,0,0.00,3.282270e+06,2987000.000,3.577539e+06,NaN
1,isFraud,int64,0,0.00,3.500000e-02,0.000,1.000000e+00,NaN
2,TransactionDT,int64,0,0.00,7.372311e+06,86400.000,1.581113e+07,NaN
3,TransactionAmt,float64,0,0.00,1.350272e+02,0.251,3.193739e+04,NaN
4,ProductCD,object,0,0.00,NaN,NaN,NaN,5.0
...,...,...,...,...,...,...,...,...
389,V335,float64,508189,86.05,5.916450e+01,0.000,5.512500e+04,NaN
390,V336,float64,508189,86.05,2.853090e+01,0.000,5.512500e+04,NaN
391,V337,float64,508189,86.05,5.535240e+01,0.000,1.040600e+05,NaN
392,V338,float64,508189,86.05,1.511605e+02,0.000,1.040600e+05,NaN


> Unique değer sayısına göre sıralı görünüm:

In [14]:
report_df_trx.sort_values(by = 'unique_values', ascending=False)

,column,dtype,missing_count,missing_ratio (%),mean,min,max,unique_values
16,R_emaildomain,object,453249,76.75,NaN,NaN,NaN,60.0
15,P_emaildomain,object,94456,15.99,NaN,NaN,NaN,59.0
4,ProductCD,object,0,0.00,NaN,NaN,NaN,5.0
8,card4,object,1577,0.27,NaN,NaN,NaN,4.0
10,card6,object,1571,0.27,NaN,NaN,NaN,4.0
...,...,...,...,...,...,...,...,...
389,V335,float64,508189,86.05,59.1645,0.0,55125.0,NaN
390,V336,float64,508189,86.05,28.5309,0.0,55125.0,NaN
391,V337,float64,508189,86.05,55.3524,0.0,104060.0,NaN
392,V338,float64,508189,86.05,151.1605,0.0,104060.0,NaN


## 5. Veri Birleştirme (Merge)

Transaction ve Identity tabloları `TransactionID` üzerinden **left join** ile birleştiriliyor.

- Left join seçildi: tüm transaction kayıtları korunur, identity eşleşmeyenler NaN alır
- Satır patlaması kontrolü yapılıyor (many-to-many riski)
- Eşleşen kayıt oranı: **%24.42** (144,233 / 590,540)

### `detect_column_types(df)`
Kolon başına anlamsal tip belirler: numeric, categorical, datetime.

In [15]:
def detect_column_types(df, unique_threshold=20, unique_ratio_threshold=0.01):
    """
    Her kolon için anlamsal tipi belirler.
    pandas dtype + unique sayısı + unique oranını birlikte kullanır.

    unique_threshold: sayısal kolonda bu kadar veya daha az unique varsa kategorik say
    unique_ratio_threshold: unique/satır oranı bunun altındaysa kategorik eğilimli
    """
    rows = []
    n = len(df)

    for col in df.columns:
        series = df[col]
        dtype = series.dtype
        n_unique = series.nunique(dropna=True)
        unique_ratio = n_unique / n if n > 0 else 0

        # İsim override: IEEE-CIS'te int ama aslında kod olan alanlar
        categorical_patterns = ("card", "addr", "ProductCD", "id_",
                                "DeviceType", "DeviceInfo", "P_emaildomain",
                                "R_emaildomain", "M")
        is_name_categorical = col.startswith(categorical_patterns)

        if is_name_categorical:
            semantic_type = "categorical (name-override)"
        elif pd.api.types.is_numeric_dtype(series):
            # Float + bol unique -> neredeyse kesin ölçülen bir nicelik
            if pd.api.types.is_float_dtype(series) and n_unique > unique_threshold:
                semantic_type = "numeric"
            elif n_unique <= unique_threshold or unique_ratio < unique_ratio_threshold:
                semantic_type = "categorical (numeric-coded)"
            else:
                semantic_type = "numeric"
        elif pd.api.types.is_datetime64_any_dtype(series):
            semantic_type = "datetime"
        else:
            semantic_type = "categorical"

        rows.append({
            "column": col,
            "pandas_dtype": str(dtype),
            "n_unique": n_unique,
            "unique_ratio (%)": round(unique_ratio * 100, 2),
            "semantic_type": semantic_type
        })

    return pd.DataFrame(rows)

### `merge_datasets(transaction, identity)`
Left join + merge raporu (satır patlaması kontrolü dahil).

In [16]:
def merge_datasets(transaction, identity, key="TransactionID", how="left"):
    """
    Transaction (ana) ve identity (opsiyonel) tablolarını birleştirir.

    Varsayılan left join: tüm transaction'lar korunur, identity'si
    olmayanlara NaN gelir (bu NaN'in kendisi sinyal olabilir).

    Join öncesi/sonrası satır sayısını ve eşleşme oranını raporlar,
    böylece sessiz satır patlamasını yakalarız.
    """
    n_before = len(transaction)

    # 1. Anahtar her iki tabloda da var mı?
    if key not in transaction.columns:
        raise KeyError(f"'{key}' transaction tablosunda yok.")
    if key not in identity.columns:
        raise KeyError(f"'{key}' identity tablosunda yok.")

    # 2. Anahtar tekilliği kontrolü (patlama riskinin kaynağı)
    tx_dup = transaction[key].duplicated().sum()
    id_dup = identity[key].duplicated().sum()

    # 3. Birleştirme
    merged = transaction.merge(identity, on=key, how=how)
    n_after = len(merged)

    # 4. Eşleşme oranı (identity ile eşleşen transaction sayısı)
    matched = transaction[key].isin(identity[key]).sum()
    match_ratio = matched / n_before * 100 if n_before > 0 else 0

    # 5. Özet rapor
    print("=" * 45)
    print("MERGE RAPORU")
    print("=" * 45)
    print(f"Transaction satır (önce) : {n_before:,}")
    print(f"Identity satır           : {len(identity):,}")
    print(f"Merged satır (sonra)     : {n_after:,}")
    print(f"Toplam kolon             : {merged.shape[1]:,}")
    print("-" * 45)
    print(f"Identity ile eşleşen tx  : {matched:,} ({match_ratio:.2f}%)")
    print(f"Transaction key duplicate: {tx_dup:,}")
    print(f"Identity key duplicate   : {id_dup:,}")

    # 6. Satır patlaması uyarısı
    if n_after > n_before:
        print("\n⚠️  UYARI: Satır sayısı arttı! Identity tarafında")
        print("    tekrar eden key var, join satırları çoğalttı.")
    else:
        print("\n✓ Satır sayısı korundu, patlama yok.")
    print("=" * 45)

    return merged

> **Merge çalıştırılıyor:**

In [17]:
merged_df = merge_datasets(trx_data, identity_data)

MERGE RAPORU
Transaction satır (önce) : 590,540
Identity satır           : 144,233
Merged satır (sonra)     : 590,540
Toplam kolon             : 434
---------------------------------------------
Identity ile eşleşen tx  : 144,233 (24.42%)
Transaction key duplicate: 0
Identity key duplicate   : 0

✓ Satır sayısı korundu, patlama yok.


## 6. Kardinalite Raporu

### `cardinality_report(df)`
Kolon başına kardinalite seviyesini belirler: `high` (>%30 unique), `mid`, `low`.

In [18]:
def cardinality_report(df,
                       high_count=100, mid_count=10,
                       high_ratio=0.3, mid_ratio=0.02,
                       categorical_only=True,
                       type_report=None):
    """
    Her kolon için cardinality'yi hem mutlak sayı hem orana göre değerlendirir.

    Seviye mantığı (sayı VEYA oran tetiklenirse üst seviyeye çıkar):
      - high : n_unique >= high_count  ya da  unique_ratio >= high_ratio
      - mid  : n_unique >= mid_count   ya da  unique_ratio >= mid_ratio
      - low  : ikisi de düşük

    categorical_only=True ve type_report verilirse, sadece kategorik
    kolonlar değerlendirilir (detect_column_types çıktısını besleyebilirsin).
    """
    n = len(df)
    rows = []

    # İstersek sadece kategorik kolonlara odaklan
    if categorical_only and type_report is not None:
        cat_cols = type_report.loc[
            type_report["semantic_type"].str.startswith("categorical"),
            "column"
        ].tolist()
        cols = [c for c in cat_cols if c in df.columns]
    else:
        cols = df.columns

    for col in cols:
        series = df[col]
        n_unique = series.nunique(dropna=True)
        unique_ratio = n_unique / n if n > 0 else 0

        # Seviye kararı: sayı veya oran, hangisi daha yüksek seviyeyi veriyorsa
        if n_unique >= high_count or unique_ratio >= high_ratio:
            level = "high"
        elif n_unique >= mid_count or unique_ratio >= mid_ratio:
            level = "mid"
        else:
            level = "low"

        rows.append({
            "column": col,
            "n_unique": n_unique,
            "unique_ratio (%)": round(unique_ratio * 100, 2),
            "cardinality_level": level
        })

    report = pd.DataFrame(rows)

    # Yüksek olanlar üste gelsin
    level_order = {"high": 0, "mid": 1, "low": 2}
    report["_sort"] = report["cardinality_level"].map(level_order)
    report = report.sort_values(["_sort", "n_unique"],
                                ascending=[True, False]).drop(columns="_sort")

    return report.reset_index(drop=True)

In [19]:
types = detect_column_types(merged_df)

In [20]:
types

,column,pandas_dtype,n_unique,unique_ratio (%),semantic_type
0,TransactionID,int64,590540,100.00,numeric
1,isFraud,int64,2,0.00,categorical (numeric-coded)
2,TransactionDT,int64,573349,97.09,numeric
3,TransactionAmt,float64,20902,3.54,numeric
4,ProductCD,object,5,0.00,categorical (name-override)
...,...,...,...,...,...
429,id_36,object,2,0.00,categorical (name-override)
430,id_37,object,2,0.00,categorical (name-override)
431,id_38,object,2,0.00,categorical (name-override)
432,DeviceType,object,2,0.00,categorical (name-override)


> **Kardinalite raporu:**

In [21]:
cardinality_report(merged_df, type_report=types)

,column,n_unique,unique_ratio (%),cardinality_level
0,id_02,115655,19.58,high
1,card1,13553,2.30,high
2,DeviceInfo,1786,0.30,high
3,id_19,522,0.09,high
4,card2,500,0.08,high
...,...,...,...,...
194,id_35,2,0.00,low
195,id_36,2,0.00,low
196,id_37,2,0.00,low
197,id_38,2,0.00,low


## 7. Sabit ve Quasi-Constant Kolonlar

### `constant_columns(df)`
Tek değerli veya neredeyse sabit (>%99 dominant) kolonları işaretler.  
Bu kolonlar model için bilgi taşımaz, feature engineering'den çıkarılabilir.

> **Not:** `isFraud` kolonu hedef değişken olduğu için koruma altına alınmıştır — sabit görünse bile filtrelenmez.

In [22]:
def constant_columns(df, quasi_threshold=0.99, target_col=None):
    """
    Sabit ve quasi-constant (neredeyse sabit) kolonları işaretler.

    - constant      : tek unique değer (NaN hariç) ya da tamamı NaN
    - quasi-constant : en sık değer, satırların quasi_threshold'undan
                       fazlasını kaplıyor
    - ok            : yeterince varyans var

    quasi_threshold : baskın değer bu oranı geçerse quasi-constant (varsayılan %99)
    target_col      : hedef değişken (örn. 'isFraud') — işaretlenir ama
                      "atma" önerisi yapılmaz, yanlışlıkla düşürülmesin diye
    """
    n = len(df)
    rows = []

    for col in df.columns:
        series = df[col]
        n_unique = series.nunique(dropna=True)

        # Baskın değerin oranı (NaN'ler hariç hesapla)
        non_null = series.dropna()
        if len(non_null) > 0:
            top_freq = non_null.value_counts(normalize=True).iloc[0]
            top_value = non_null.value_counts().index[0]
        else:
            top_freq = 1.0          # tamamı NaN -> tam sabit gibi
            top_value = None

        # Sınıflandırma
        if n_unique <= 1:
            flag = "constant"
        elif top_freq >= quasi_threshold:
            flag = "quasi-constant"
        else:
            flag = "ok"

        # Öneri: target ise asla atma deme
        if target_col is not None and col == target_col:
            recommendation = "KEEP (target)"
        elif flag == "constant":
            recommendation = "drop"
        elif flag == "quasi-constant":
            recommendation = "review"
        else:
            recommendation = "keep"

        rows.append({
            "column": col,
            "n_unique": n_unique,
            "top_value": top_value,
            "top_freq (%)": round(top_freq * 100, 2),
            "flag": flag,
            "recommendation": recommendation
        })

    report = pd.DataFrame(rows)

    # Sorunlu olanlar üste gelsin
    flag_order = {"constant": 0, "quasi-constant": 1, "ok": 2}
    report["_sort"] = report["flag"].map(flag_order)
    report = report.sort_values(["_sort", "top_freq (%)"],
                                ascending=[True, False]).drop(columns="_sort")

    return report.reset_index(drop=True)

In [23]:
constant_columns(merged_df, target_col="isFraud")

,column,n_unique,top_value,top_freq (%),flag,recommendation
0,V305,2,1.0,100.00,quasi-constant,review
1,M1,2,T,99.99,quasi-constant,review
2,V1,2,1.0,99.99,quasi-constant,review
3,V65,2,1.0,99.97,quasi-constant,review
4,V241,5,1.0,99.97,quasi-constant,review
...,...,...,...,...,...,...
429,card1,13553,7919,2.53,ok,keep
430,D8,12353,0.791666,1.80,ok,keep
431,id_02,115655,1102.0,0.01,ok,keep
432,TransactionID,590540,2987000,0.00,ok,keep


## 8. Duplikasyon Raporu

### `duplicate_report(df)`
İki tür duplikasyonu ayrı raporlar:
1. **Key duplicate:** `TransactionID` tekrar ediyor mu?
2. **Tam satır duplicate:** Tüm değerler aynı satırlar var mı?

> **Bulgu:** TransactionID tekrarı yok ✓. 3 tam satır tekrarı var — fraud'da tekrarlayan işlem anlamlı sinyal olabilir, silinmedi.

In [24]:
def duplicate_report(df, key="TransactionID", subset=None):
    """
    İki tür duplicate'i ayrı ayrı raporlar:

    1) Key duplicate     : anahtar (TransactionID) tekrar ediyor mu?
                           Tekilse 0 bekleriz; >0 ise veri/join sorunu.
    2) Tam satır duplicate: key hariç tüm kolonları aynı olan satırlar.
                           Fraud'da 'tekrarlayan işlem' sinyali olabilir,
                           o yüzden silinmez, işaretlenir.

    subset : tam duplicate kontrolünde bakılacak kolonlar.
             Verilmezse key dışındaki tüm kolonlar kullanılır.
    """
    n = len(df)

    # 1) KEY DUPLICATE
    if key in df.columns:
        key_dup_count = df[key].duplicated().sum()
        key_unique = df[key].nunique()
    else:
        key_dup_count = None
        key_unique = None

    # 2) TAM SATIR DUPLICATE (key hariç)
    if subset is None:
        subset = [c for c in df.columns if c != key]

    full_dup_mask = df.duplicated(subset=subset, keep=False)  # tüm kopyalar işaretli
    full_dup_count = df.duplicated(subset=subset).sum()        # ilk hariç fazlalık

    # Özet rapor
    print("=" * 45)
    print("DUPLICATE RAPORU")
    print("=" * 45)
    print(f"Toplam satır                : {n:,}")
    print("-" * 45)
    print("KEY DUPLICATE")
    if key_dup_count is not None:
        print(f"  '{key}' tekrar eden       : {key_dup_count:,}")
        print(f"  '{key}' unique sayısı     : {key_unique:,}")
        if key_dup_count == 0:
            print("  ✓ Anahtar tekil, sorun yok.")
        else:
            print("  ⚠️  Anahtar tekrar ediyor! Veri/join sorunu.")
    else:
        print(f"  '{key}' kolonu tabloda yok.")
    print("-" * 45)
    print("TAM SATIR DUPLICATE (key hariç)")
    print(f"  Fazlalık satır (ilk hariç): {full_dup_count:,}")
    print(f"  Duplicate'e dahil satır   : {full_dup_mask.sum():,}")
    if full_dup_count == 0:
        print("  ✓ Tam tekrar eden satır yok.")
    else:
        print("  ⚠️  Tekrar eden satırlar var — silmeden incele")
        print("      (fraud'da tekrarlayan işlem sinyali olabilir).")
    print("=" * 45)

    # Duplicate satırları döndür ki incelenebilsin
    duplicated_rows = df[full_dup_mask].sort_values(subset) if full_dup_count > 0 else df.iloc[0:0]

    return duplicated_rows

In [25]:
dups = duplicate_report(merged_df)

DUPLICATE RAPORU
Toplam satır                : 590,540
---------------------------------------------
KEY DUPLICATE
  'TransactionID' tekrar eden       : 0
  'TransactionID' unique sayısı     : 590,540
  ✓ Anahtar tekil, sorun yok.
---------------------------------------------
TAM SATIR DUPLICATE (key hariç)
  Fazlalık satır (ilk hariç): 3
  Duplicate'e dahil satır   : 6
  ⚠️  Tekrar eden satırlar var — silmeden incele
      (fraud'da tekrarlayan işlem sinyali olabilir).


## 9. Geçerlilik Raporu

### `validity_report(df, rules=None)`
Değerlerin mantıksal geçerliliğini iki katmanda kontrol eder:
1. **Otomatik:** Inf değerler, negatif sayılar (negatif olmaması beklenen kolonlar)
2. **Kural bazlı:** `rules` dict ile özel min/max/allow_negative kısıtları

Burada `TransactionAmt >= 0` ve `TransactionDT >= 0` kuralları uygulandı.

In [26]:
def validity_report(df, rules=None, check_negative=True):
    """
    Değerlerin mantıksal geçerliliğini kontrol eder.

    İki katman:
      1) Otomatik kontroller (domain bilgisi gerektirmez):
         - sonsuz (inf) değerler
         - (opsiyonel) sayısal kolonlarda negatif değerler
      2) Kullanıcı kuralları (rules): kolon bazında domain kısıtları

    rules formatı (hepsi opsiyonel):
      {
        "TransactionAmt": {"min": 0, "allow_negative": False},
        "age":            {"min": 0, "max": 120},
        "prob_score":     {"min": 0, "max": 1},
        "TransactionDT":  {"max": 15811131},   # mantıklı üst sınır
      }

    check_negative : True ise TÜM sayısal kolonlarda negatif sayar
                     (bilgilendirme amaçlı; kural değil, sadece işaret)
    """
    rules = rules or {}
    n = len(df)
    rows = []

    for col in df.columns:
        series = df[col]
        is_numeric = pd.api.types.is_numeric_dtype(series)

        issues = []
        n_inf = 0
        n_negative = 0
        n_rule_violation = 0

        if is_numeric:
            # --- Katman 1: otomatik ---
            n_inf = np.isinf(series.replace([np.inf, -np.inf], np.inf)
                             .where(series.isin([np.inf, -np.inf]), np.nan)).sum()
            # daha sade ve güvenli yol:
            n_inf = np.isinf(series.to_numpy(dtype="float64", na_value=np.nan)).sum()

            if n_inf > 0:
                issues.append(f"{n_inf} inf")

            if check_negative:
                n_negative = (series < 0).sum()
                if n_negative > 0:
                    issues.append(f"{n_negative} negatif")

            # --- Katman 2: kurallar ---
            if col in rules:
                r = rules[col]
                if "min" in r:
                    v = (series < r["min"]).sum()
                    if v > 0:
                        issues.append(f"{v} < min({r['min']})")
                        n_rule_violation += v
                if "max" in r:
                    v = (series > r["max"]).sum()
                    if v > 0:
                        issues.append(f"{v} > max({r['max']})")
                        n_rule_violation += v
                if r.get("allow_negative") is False:
                    v = (series < 0).sum()
                    if v > 0 and "negatif" not in " ".join(issues):
                        issues.append(f"{v} negatif (kural)")
                        n_rule_violation += v

        rows.append({
            "column": col,
            "is_numeric": is_numeric,
            "n_inf": int(n_inf),
            "n_negative": int(n_negative),
            "n_rule_violation": int(n_rule_violation),
            "issues": "; ".join(issues) if issues else "ok"
        })

    report = pd.DataFrame(rows)

    # Sorunlu olanlar üste
    report["_has_issue"] = (report["issues"] != "ok").astype(int)
    report = report.sort_values(["_has_issue", "n_rule_violation"],
                                ascending=[False, False]).drop(columns="_has_issue")

    return report.reset_index(drop=True)

In [27]:
rules = {
    "TransactionAmt": {"min": 0, "allow_negative": False},
    "TransactionDT":  {"min": 0},
}
validity_report(merged_df, rules=rules)

,column,is_numeric,n_inf,n_negative,n_rule_violation,issues
0,D4,True,0,15,0,15 negatif
1,D6,True,0,3,0,3 negatif
2,D11,True,0,7,0,7 negatif
3,D12,True,0,2,0,2 negatif
4,D14,True,0,3,0,3 negatif
...,...,...,...,...,...,...
429,id_36,False,0,0,0,ok
430,id_37,False,0,0,0,ok
431,id_38,False,0,0,0,ok
432,DeviceType,False,0,0,0,ok


## 10. Outlier Raporu

### `outlier_report(df)`
IQR yöntemiyle uç değerleri tespit eder. **Silmez, işaretler.**

> **Tasarım kararı:** Fraud sinyali genellikle kuyrukta yaşar (anormal tutar, anormal hız).  
> Outlier = sinyal; silmek model kalitesini düşürür.

In [28]:
def outlier_report(df, k=1.5, type_report=None, columns=None):
    """
    IQR yöntemiyle uç değerleri TESPİT EDER — silmez, doldurmaz.

    Fraud'da outlier genelde sinyaldir (anormal tutar/davranış),
    o yüzden bu fonksiyon yalnızca işaretler ve sayar.

    Sınırlar:
        IQR = Q3 - Q1
        alt sınır = Q1 - k*IQR
        üst sınır = Q3 + k*IQR

    type_report : detect_column_types çıktısı verilirse, sadece GERÇEK
                  sayısal kolonlara bakar (numeric-coded ID'leri eler).
    columns     : elle kolon listesi vermek istersen.
    """
    n = len(df)
    rows = []

    # Hangi kolonlara bakılacak?
    if columns is not None:
        num_cols = [c for c in columns if c in df.columns]
    elif type_report is not None:
        num_cols = type_report.loc[
            type_report["semantic_type"] == "numeric", "column"
        ].tolist()
        num_cols = [c for c in num_cols if c in df.columns]
    else:
        num_cols = df.select_dtypes(include="number").columns.tolist()

    for col in num_cols:
        series = df[col].dropna()
        if len(series) == 0:
            continue

        q1 = series.quantile(0.25)
        q3 = series.quantile(0.75)
        iqr = q3 - q1

        lower = q1 - k * iqr
        upper = q3 + k * iqr

        n_lower = (series < lower).sum()
        n_upper = (series > upper).sum()
        n_outlier = n_lower + n_upper
        outlier_ratio = n_outlier / n * 100 if n > 0 else 0

        rows.append({
            "column": col,
            "Q1": round(q1, 4),
            "Q3": round(q3, 4),
            "IQR": round(iqr, 4),
            "lower_bound": round(lower, 4),
            "upper_bound": round(upper, 4),
            "n_outlier_low": int(n_lower),
            "n_outlier_high": int(n_upper),
            "n_outlier_total": int(n_outlier),
            "outlier_ratio (%)": round(outlier_ratio, 2)
        })

    report = pd.DataFrame(rows)
    if not report.empty:
        report = report.sort_values("outlier_ratio (%)", ascending=False).reset_index(drop=True)

    return report

In [29]:
outlier_report(merged_df, type_report=types)

,column,Q1,Q3,IQR,lower_bound,upper_bound,n_outlier_low,n_outlier_high,n_outlier_total,outlier_ratio (%)
0,C8,0.000000e+00,0.000000e+00,0.000,0.000000e+00,0.000000e+00,0,142873,142873,24.19
1,V303,0.000000e+00,0.000000e+00,0.000,0.000000e+00,0.000000e+00,0,141029,141029,23.88
2,C4,0.000000e+00,0.000000e+00,0.000,0.000000e+00,0.000000e+00,0,138657,138657,23.48
3,C10,0.000000e+00,0.000000e+00,0.000,0.000000e+00,0.000000e+00,0,137098,137098,23.22
4,V128,0.000000e+00,0.000000e+00,0.000,0.000000e+00,0.000000e+00,0,136013,136013,23.03
...,...,...,...,...,...,...,...,...,...,...
230,D9,2.083000e-01,8.333000e-01,0.625,-7.292000e-01,1.770800e+00,0,0,0,0.00
231,D11,0.000000e+00,2.740000e+02,274.000,-4.110000e+02,6.850000e+02,0,0,0,0.00
232,D2,2.600000e+01,2.760000e+02,250.000,-3.490000e+02,6.510000e+02,0,0,0,0.00
233,TransactionDT,3.027058e+06,1.124662e+07,8219562.250,-9.302286e+06,2.357596e+07,0,0,0,0.00


## 11. Dağılım Raporu

### `distribution_report(df)`
Sayısal kolonlar için çarpıklık (skewness) ve basıklık (kurtosis) hesaplar.  
Log transform adaylarını işaretler. Kategorik kolonlar için baskınlık analizi yapar.

In [30]:
def distribution_report(df, type_report=None, top_n=3):
    """
    Kolonların dağılım ŞEKLİNİ analiz eder.

    Sayısal kolonlar:
        - skewness (çarpıklık): pozitif = sağa kuyruk -> log dönüşümü adayı
        - kurtosis (basıklık) : yüksek = ağır kuyruk / çok uç değer
        - median, std
    Kategorik kolonlar:
        - en sık değer ve oranı
        - ilk top_n değerin toplam kapsama oranı (denge ölçüsü)

    type_report : detect_column_types çıktısı. Verilirse numeric vs
                  categorical ayrımı semantik tipe göre yapılır
                  (numeric-coded ID'ler kategorik tarafta değerlendirilir).
    """
    n = len(df)

    # Tip ayrımı
    if type_report is not None:
        numeric_cols = type_report.loc[
            type_report["semantic_type"] == "numeric", "column"
        ].tolist()
        cat_cols = type_report.loc[
            type_report["semantic_type"].str.startswith("categorical"), "column"
        ].tolist()
    else:
        numeric_cols = df.select_dtypes(include="number").columns.tolist()
        cat_cols = [c for c in df.columns if c not in numeric_cols]

    numeric_cols = [c for c in numeric_cols if c in df.columns]
    cat_cols = [c for c in cat_cols if c in df.columns]

    # --- SAYISAL DAĞILIM ---
    num_rows = []
    for col in numeric_cols:
        s = df[col].dropna()
        if len(s) == 0:
            continue
        skew = s.skew()
        num_rows.append({
            "column": col,
            "median": round(s.median(), 4),
            "std": round(s.std(), 4),
            "skewness": round(skew, 4),
            "kurtosis": round(s.kurtosis(), 4),
            "skew_flag": ("high right-skew" if skew > 1
                          else "high left-skew" if skew < -1
                          else "near-symmetric"),
            "log_transform_candidate": (skew > 1 and (s >= 0).all())
        })
    numeric_distribution = pd.DataFrame(num_rows)
    if not numeric_distribution.empty:
        numeric_distribution = numeric_distribution.reindex(
            numeric_distribution["skewness"].abs().sort_values(ascending=False).index
        ).reset_index(drop=True)

    # --- KATEGORİK DAĞILIM ---
    cat_rows = []
    for col in cat_cols:
        s = df[col].dropna()
        if len(s) == 0:
            continue
        vc = s.value_counts(normalize=True)
        top_value = vc.index[0]
        top_ratio = vc.iloc[0]
        topn_coverage = vc.iloc[:top_n].sum()
        cat_rows.append({
            "column": col,
            "n_unique": s.nunique(),
            "top_value": top_value,
            "top_ratio (%)": round(top_ratio * 100, 2),
            f"top{top_n}_coverage (%)": round(topn_coverage * 100, 2),
            "balance_flag": ("dominated" if top_ratio > 0.9
                             else "imbalanced" if top_ratio > 0.5
                             else "spread")
        })
    categorical_distribution = pd.DataFrame(cat_rows)
    if not categorical_distribution.empty:
        categorical_distribution = categorical_distribution.sort_values(
            "top_ratio (%)", ascending=False
        ).reset_index(drop=True)

    return numeric_distribution, categorical_distribution

In [31]:
num_dist, cat_dist = distribution_report(merged_df, type_report=types)

> **Sayısal dağılım** (skewness'e göre sıralı):

In [32]:
num_dist   # skewness'e göre sıralı sayısal dağılım

,column,median,std,skewness,kurtosis,skew_flag,log_transform_candidate
0,V311,0.000000e+00,1.023749e+02,323.8314,156542.4169,high right-skew,True
1,V129,0.000000e+00,1.138328e+02,240.2743,102763.1368,high right-skew,True
2,V309,0.000000e+00,1.162543e+02,224.8753,94325.7126,high right-skew,True
3,V206,0.000000e+00,1.914740e+02,207.8816,54107.1017,high right-skew,True
4,V319,0.000000e+00,3.323048e+02,181.8337,47649.8999,high right-skew,True
...,...,...,...,...,...,...,...
230,D2,9.700000e+01,1.773159e+02,1.0157,-0.1686,high right-skew,True
231,D15,5.200000e+01,2.027267e+02,0.9574,-0.5279,near-symmetric,False
232,D9,6.667000e-01,3.169000e-01,-0.5859,-1.1395,near-symmetric,False
233,TransactionDT,7.306528e+06,4.617224e+06,0.1312,-1.2291,near-symmetric,False


> **Kategorik dağılım** (baskınlığa göre sıralı):

In [33]:
cat_dist   # baskınlığa göre sıralı kategorik dağılım

,column,n_unique,top_value,top_ratio (%),top3_coverage (%),balance_flag
0,V305,2,1.0,100.00,100.00,dominated
1,V1,2,1.0,99.99,100.00,dominated
2,M1,2,T,99.99,100.00,dominated
3,V65,2,1.0,99.97,100.00,dominated
4,V241,5,1.0,99.97,100.00,dominated
...,...,...,...,...,...,...
194,addr1,332,299.0,8.83,24.98,spread
195,card2,500,321.0,8.41,23.40,spread
196,id_07,84,0.0,7.93,17.11,spread
197,card1,13553,7919,2.53,6.68,spread


## 12. Kalite Özeti

### `quality_summary(df)`
Tüm alt raporları birleştirir. Her kolona 🟢/🟡/🔴 bayrak atar:
- 🔴 Yüksek eksik veri (>%70)
- 🟡 Orta eksik veri (%30–70) veya quasi-constant
- 🟢 Temiz

**Çalıştırma sırası:** merge → types → (missing / card / const / outlier / dist) → quality_summary

In [34]:
def quality_summary(df,
                    type_report=None,
                    cardinality=None,
                    constant=None,
                    outlier=None,
                    high_missing=90.0,
                    mid_missing=20.0,
                    target_col=None):
    """
    Tüm kalite teşhislerini kolon ekseninde tek tabloda birleştirir
    ve her kolona bütünsel bir kalite bayrağı (🟢/🟡/🔴) verir.

    Daha önce ürettiğin raporları parametre olarak alır (yeniden
    hesaplamaz). Verilmeyenler atlanır.

    Bayrak (en kötü sorun rengi belirler):
      🔴 : constant  veya  missing > high_missing
      🟡 : quasi-constant, high cardinality, missing > mid_missing,
           ya da outlier mevcut
      🟢 : temiz
    """
    n = len(df)

    # Temel iskelet: her kolon bir satır
    summary = pd.DataFrame({"column": df.columns})

    # --- Eksik veri (her zaman hesaplanır, ucuz) ---
    miss = df.isnull().sum()
    summary["missing_ratio (%)"] = summary["column"].map(
        (miss / n * 100).round(2)
    )

    # --- Tip ---
    if type_report is not None:
        t = type_report.set_index("column")["semantic_type"]
        summary["semantic_type"] = summary["column"].map(t)

    # --- Cardinality ---
    if cardinality is not None:
        c = cardinality.set_index("column")["cardinality_level"]
        summary["cardinality_level"] = summary["column"].map(c)

    # --- Constant / quasi-constant ---
    if constant is not None:
        f = constant.set_index("column")["flag"]
        summary["constant_flag"] = summary["column"].map(f)

    # --- Outlier (varlık + oran) ---
    if outlier is not None and not outlier.empty:
        o = outlier.set_index("column")["outlier_ratio (%)"]
        summary["outlier_ratio (%)"] = summary["column"].map(o)

    # --- Bütünsel bayrak ---
    def decide_flag(row):
        col = row["column"]
        # Target asla kırmızıya boyanıp atılma listesine düşmesin
        if target_col is not None and col == target_col:
            return "🟢 (target)"

        miss_r = row.get("missing_ratio (%)", 0)
        const_f = row.get("constant_flag", None)
        card_l = row.get("cardinality_level", None)
        out_r = row.get("outlier_ratio (%)", 0)

        # 🔴 ciddi
        if const_f == "constant":
            return "🔴"
        if pd.notna(miss_r) and miss_r > high_missing:
            return "🔴"

        # 🟡 dikkat
        if const_f == "quasi-constant":
            return "🟡"
        if card_l == "high":
            return "🟡"
        if pd.notna(miss_r) and miss_r > mid_missing:
            return "🟡"
        if pd.notna(out_r) and out_r > 0:
            return "🟡"

        return "🟢"

    summary["quality_flag"] = summary.apply(decide_flag, axis=1)

    # Sorunlular üste: 🔴 -> 🟡 -> 🟢
    flag_rank = {"🔴": 0, "🟡": 1, "🟢": 2, "🟢 (target)": 3}
    summary["_rank"] = summary["quality_flag"].map(
        lambda x: flag_rank.get(x, 2)
    )
    summary = summary.sort_values(
        ["_rank", "missing_ratio (%)"], ascending=[True, False]
    ).drop(columns="_rank").reset_index(drop=True)

    return summary

In [35]:
types = detect_column_types(merged_df)
card = cardinality_report(merged_df, type_report=types)
const = constant_columns(merged_df, target_col="isFraud")
outl = outlier_report(merged_df, type_report=types)

In [36]:
summary = quality_summary(
    merged_df,
    type_report=types,
    cardinality=card,
    constant=const,
    outlier=outl,
    target_col="isFraud"
)

In [37]:
summary

,column,missing_ratio (%),semantic_type,cardinality_level,constant_flag,outlier_ratio (%),quality_flag
0,id_24,99.20,categorical (name-override),mid,ok,NaN,🔴
1,id_07,99.13,categorical (name-override),mid,ok,NaN,🔴
2,id_08,99.13,categorical (name-override),mid,ok,NaN,🔴
3,id_21,99.13,categorical (name-override),high,ok,NaN,🔴
4,id_25,99.13,categorical (name-override),high,ok,NaN,🔴
...,...,...,...,...,...,...,...
429,V286,0.00,categorical (numeric-coded),low,ok,NaN,🟢
430,V297,0.00,categorical (numeric-coded),mid,ok,NaN,🟢
431,V302,0.00,categorical (numeric-coded),mid,ok,NaN,🟢
432,V304,0.00,categorical (numeric-coded),mid,ok,NaN,🟢


## 13. Nadir Kombinasyon Raporu

### `rare_combination_report(df)`
Düşük-orta kardinaliteli kategorik kolonları **ikişerli çaprazlar**.  
Her kombinasyon için `fraud_rate` ve `fraud_lift` hesaplar.

> **Amaç:** Tek başına anlamsız görünen ama birlikte yüksek fraud oranı gösteren değer çiftlerini bulmak.  
> Örnek: *"american express + belirli bir V değeri"* → %100 fraud rate.

In [38]:
def rare_combination_report(df, type_report=None,
                            min_unique=3, max_unique=50,
                            max_columns=12,
                            rare_threshold=0.001,
                            top_n=20,
                            target_col=None):
    """
    Düşük-orta cardinality kategorik kolonları ikişerli çaprazlayıp
    NADİR görülen değer kombinasyonlarını tespit eder.

    Mantık: tek tek yaygın olan iki değerin BİRLİKTE nadir görülmesi
    fraud sinyali olabilir (örn. yaygın ProductCD + yaygın card4,
    ama bu ikisi birlikte neredeyse hiç görülmüyor).

    min_unique / max_unique : çaprazlanacak kolonların cardinality aralığı
                              (çok düşük = anlamsız, çok yüksek = patlar)
    max_columns : güvenlik freni — bu kadar kolondan fazlasını çaprazlama
    rare_threshold : kombinasyon frekansı bu oranın altındaysa 'rare'
    top_n : her kolon çifti için en nadir kaç kombinasyon raporlansın
    """
    n = len(df)

    # Genel fraud oranı (baz çizgi) — rare kombinasyonların fraud'unu
    # bununla kıyaslayacağız
    if target_col is not None and target_col in df.columns:
        global_fraud_rate = df[target_col].mean()
        print(f"Genel {target_col} oranı: {global_fraud_rate*100:.4f}%")
    else:
        global_fraud_rate = None

    # 1) Aday kategorik kolonları seç
    if type_report is not None:
        cat_cols = type_report.loc[
            type_report["semantic_type"].str.startswith("categorical"), "column"
        ].tolist()
    else:
        cat_cols = df.select_dtypes(include=["object", "category"]).columns.tolist()

    # 2) Cardinality aralığına göre filtrele
    eligible = []
    for col in cat_cols:
        if col not in df.columns:
            continue
        nu = df[col].nunique(dropna=True)
        if min_unique <= nu <= max_unique:
            eligible.append((col, nu))

    # En düşük cardinality'den başla (en yaygın/anlamlı kombinasyonlar)
    eligible = sorted(eligible, key=lambda x: x[1])[:max_columns]
    eligible_cols = [c for c, _ in eligible]

    if len(eligible_cols) < 2:
        print("Çaprazlanacak yeterli uygun kategorik kolon yok.")
        return pd.DataFrame()

    print(f"Çaprazlanan kolonlar ({len(eligible_cols)}): {eligible_cols}")

    # 3) İkişerli kombinasyonları tara
    from itertools import combinations
    rows = []

    for col_a, col_b in combinations(eligible_cols, 2):
        # target varsa onu da çek ki kombinasyon başına fraud oranı hesaplayalım
        cols_needed = [col_a, col_b]
        if global_fraud_rate is not None:
            cols_needed = cols_needed + [target_col]

        pair = df[cols_needed].dropna(subset=[col_a, col_b])
        if len(pair) == 0:
            continue

        # Birlikte görülme frekansı
        combo_counts = pair.groupby([col_a, col_b]).size()
        combo_ratio = combo_counts / len(pair)

        # Nadir olanları al
        rare = combo_ratio[combo_ratio < rare_threshold].sort_values()
        rare = rare.head(top_n)

        # target varsa kombinasyon başına fraud oranı
        if global_fraud_rate is not None:
            combo_fraud = pair.groupby([col_a, col_b])[target_col].mean()

        for (val_a, val_b), ratio in rare.items():
            row = {
                "col_a": col_a,
                "col_b": col_b,
                "value_a": val_a,
                "value_b": val_b,
                "count": int(combo_counts.loc[(val_a, val_b)]),
                "combo_ratio (%)": round(ratio * 100, 4)
            }
            if global_fraud_rate is not None:
                fr = combo_fraud.loc[(val_a, val_b)]
                row["fraud_rate (%)"] = round(fr * 100, 4)
                # bu kombinasyon genel orandan kaç kat daha fraudlu?
                row["fraud_lift"] = round(fr / global_fraud_rate, 2) if global_fraud_rate > 0 else np.nan
            rows.append(row)

    report = pd.DataFrame(rows)
    if not report.empty:
        if "fraud_lift" in report.columns:
            # Hem nadir HEM yüksek fraud oranlı olanlar en üstte
            report = report.sort_values("fraud_lift", ascending=False).reset_index(drop=True)
        else:
            report = report.sort_values("combo_ratio (%)").reset_index(drop=True)

    return report

In [39]:
rare_combos = rare_combination_report(merged_df, type_report=types, target_col='isFraud')

Genel isFraud oranı: 3.4990%
Çaprazlanan kolonlar (12): ['M4', 'V68', 'V89', 'V94', 'id_15', 'id_23', 'card4', 'card6', 'V12', 'V27', 'V28', 'V35']


In [40]:
rare_combos

,col_a,col_b,value_a,value_b,count,combo_ratio (%),fraud_rate (%),fraud_lift
0,M4,card4,M2,american express,2,0.0006,100.0000,28.58
1,V94,card4,1.0,american express,2,0.0004,100.0000,28.58
2,V12,V35,3.0,0.0,4,0.0010,50.0000,14.29
3,V94,V12,1.0,2.0,220,0.0444,31.3636,8.96
4,V94,card4,1.0,discover,58,0.0116,27.5862,7.88
...,...,...,...,...,...,...,...,...
180,V89,V12,2.0,0.0,30,0.0061,0.0000,0.00
181,V89,V12,1.0,0.0,291,0.0587,0.0000,0.00
182,V89,V27,2.0,2.0,5,0.0010,0.0000,0.00
183,V89,V27,0.0,1.0,11,0.0022,0.0000,0.00


## 14. Kolon İlişki Raporu

### `column_relationship_report(df)`
Üç tablo döndürür:
1. **numeric_corr:** Yüksek korelasyonlu numeric çiftler (>0.9) — V grubu kolonları burada yoğunlaşır
2. **cat_dependency:** Kategorik determinism — M1/M2/M3 → ProductCD ilişkisi gibi
3. **target_power:** Her kolonun `isFraud` ile ayırt edici gücü — min/max fraud rate aralığı

> **Önemli bulgu:** V153, V154, V194, V195 kolonları %3 → %98+ fraud rate aralığına sahip.  
> Bu kolonlar feature engineering'de öncelikli ele alınmalıdır.

In [41]:
def column_relationship_report(df, type_report=None, target_col="isFraud",
                               corr_threshold=0.9,
                               max_numeric=80,
                               dep_threshold=0.95,
                               max_categorical=15):
    """
    Kolonlar arası ilişkileri üç açıdan keşfeder, ÜÇ AYRI tablo döndürür:

      1) numeric_corr : yüksek korelasyonlu numeric çiftler (redundancy)
      2) cat_dependency: kategorik fonksiyonel bağımlılık (A -> B)
      3) target_power : kategoriklerin fraud'u ayırt etme gücü

    corr_threshold  : bu eşiğin üstündeki numeric çiftler raporlanır
    max_numeric     : korelasyon için en fazla kaç numeric kolon (patlama freni)
    dep_threshold   : A->B bağımlılığı bu oranı geçerse 'güçlü'
    max_categorical : bağımlılık taramasında en fazla kaç kategorik kolon
    """
    n = len(df)

    # Tip ayrımı
    if type_report is not None:
        numeric_cols = type_report.loc[
            type_report["semantic_type"] == "numeric", "column"].tolist()
        cat_cols = type_report.loc[
            type_report["semantic_type"].str.startswith("categorical"), "column"].tolist()
    else:
        numeric_cols = df.select_dtypes(include="number").columns.tolist()
        cat_cols = [c for c in df.columns if c not in numeric_cols]

    numeric_cols = [c for c in numeric_cols if c in df.columns and c != target_col]
    cat_cols = [c for c in cat_cols if c in df.columns and c != target_col]

    # ============ BLOK 1: NUMERIC KORELASYON ============
    # Patlama freni: çok fazla numeric varsa en çok dolu olanları al
    if len(numeric_cols) > max_numeric:
        fill = df[numeric_cols].notna().sum().sort_values(ascending=False)
        numeric_cols_corr = fill.head(max_numeric).index.tolist()
    else:
        numeric_cols_corr = numeric_cols

    corr_rows = []
    if len(numeric_cols_corr) >= 2:
        corr_matrix = df[numeric_cols_corr].corr().abs()
        # Sadece üst üçgen (her çift bir kez)
        import numpy as np
        upper = corr_matrix.where(
            np.triu(np.ones(corr_matrix.shape), k=1).astype(bool))

        for col_a in upper.columns:
            for col_b in upper.index:
                val = upper.loc[col_b, col_a]
                if pd.notna(val) and val >= corr_threshold:
                    corr_rows.append({
                        "col_a": col_a,
                        "col_b": col_b,
                        "abs_corr": round(val, 4)
                    })
    numeric_corr = pd.DataFrame(corr_rows)
    if not numeric_corr.empty:
        numeric_corr = numeric_corr.sort_values("abs_corr", ascending=False).reset_index(drop=True)

    # ============ BLOK 2: KATEGORİK FONKSİYONEL BAĞIMLILIK ============
    # Çok yüksek cardinality kategorikleri ele (patlama)
    dep_candidates = [c for c in cat_cols
                      if df[c].nunique(dropna=True) <= 1000][:max_categorical]

    dep_rows = []
    from itertools import permutations  # A->B ve B->A farklı, o yüzden permutations
    for col_a, col_b in permutations(dep_candidates, 2):
        sub = df[[col_a, col_b]].dropna()
        if len(sub) == 0:
            continue
        # A'nın her değeri için B kaç farklı değer alıyor?
        b_per_a = sub.groupby(col_a)[col_b].nunique()
        # A'nın değerlerinin kaçı B'de tek değere eşleniyor?
        single_ratio = (b_per_a == 1).mean()
        if single_ratio >= dep_threshold:
            dep_rows.append({
                "determinant (A)": col_a,
                "dependent (B)": col_b,
                "A->B_strength": round(single_ratio, 4),
                "note": "B, A tarafından belirleniyor (B redundant olabilir)"
            })
    cat_dependency = pd.DataFrame(dep_rows)
    if not cat_dependency.empty:
        cat_dependency = cat_dependency.sort_values("A->B_strength", ascending=False).reset_index(drop=True)

    # ============ BLOK 3: TARGET AYIRT EDİCİLİK ============
    power_rows = []
    if target_col in df.columns:
        global_rate = df[target_col].mean()
        for col in cat_cols:
            # Çok yüksek cardinality'de anlamsız, ele
            if df[col].nunique(dropna=True) > 1000:
                continue
            grp = df.groupby(col)[target_col].agg(["mean", "size"])
            # Çok küçük grupları gürültü saymamak için min destek
            grp = grp[grp["size"] >= 50]
            if len(grp) < 2:
                continue
            spread = grp["mean"].max() - grp["mean"].min()
            max_lift = grp["mean"].max() / global_rate if global_rate > 0 else np.nan
            power_rows.append({
                "column": col,
                "n_groups": len(grp),
                "min_fraud_rate (%)": round(grp["mean"].min() * 100, 4),
                "max_fraud_rate (%)": round(grp["mean"].max() * 100, 4),
                "fraud_spread (%)": round(spread * 100, 4),
                "max_lift": round(max_lift, 2)
            })
    target_power = pd.DataFrame(power_rows)
    if not target_power.empty:
        target_power = target_power.sort_values("fraud_spread (%)", ascending=False).reset_index(drop=True)

    return numeric_corr, cat_dependency, target_power

In [42]:
num_corr, cat_dep, tgt_power = column_relationship_report(merged_df, type_report=types)

num_corr    # yüksek korelasyonlu numeric çiftler (V-grupları burada patlar)
cat_dep     # card1->card2 gibi fonksiyonel bağımlılıklar
tgt_power   # fraud'u en çok ayıran kategorikler

,column,n_groups,min_fraud_rate (%),max_fraud_rate (%),fraud_spread (%),max_lift
0,V154,5,2.9828,98.7654,95.7826,28.23
1,V153,5,3.0026,98.3333,95.3308,28.10
2,V195,6,5.8252,100.0000,94.1748,28.58
3,V194,5,5.9631,100.0000,94.0369,28.58
4,V197,6,5.9631,100.0000,94.0369,28.58
...,...,...,...,...,...,...
185,M5,2,2.6523,3.7697,1.1175,1.08
186,M6,2,1.7044,2.3686,0.6642,0.68
187,M8,2,1.6218,2.1726,0.5508,0.62
188,V88,2,2.9101,3.2687,0.3586,0.93


In [43]:
num_corr

,col_a,col_b,abs_corr
0,V95,V101,0.9996
1,V279,V293,0.9996
2,C7,C12,0.9995
3,V101,V293,0.9989
4,V97,V103,0.9988
...,...,...,...
308,V279,V307,0.9028
309,V95,V317,0.9026
310,V103,V294,0.9014
311,V101,V317,0.9011


In [44]:
cat_dep

,determinant (A),dependent (B),A->B_strength,note
0,M1,ProductCD,1.0000,"B, A tarafından belirleniyor (B redundant olab..."
1,M2,ProductCD,1.0000,"B, A tarafından belirleniyor (B redundant olab..."
2,M3,ProductCD,1.0000,"B, A tarafından belirleniyor (B redundant olab..."
3,M5,ProductCD,1.0000,"B, A tarafından belirleniyor (B redundant olab..."
4,card3,M1,0.9792,"B, A tarafından belirleniyor (B redundant olab..."
5,card2,card3,0.9640,"B, A tarafından belirleniyor (B redundant olab..."
6,card2,M1,0.9616,"B, A tarafından belirleniyor (B redundant olab..."


In [45]:
tgt_power

,column,n_groups,min_fraud_rate (%),max_fraud_rate (%),fraud_spread (%),max_lift
0,V154,5,2.9828,98.7654,95.7826,28.23
1,V153,5,3.0026,98.3333,95.3308,28.10
2,V195,6,5.8252,100.0000,94.1748,28.58
3,V194,5,5.9631,100.0000,94.0369,28.58
4,V197,6,5.9631,100.0000,94.0369,28.58
...,...,...,...,...,...,...
185,M5,2,2.6523,3.7697,1.1175,1.08
186,M6,2,1.7044,2.3686,0.6642,0.68
187,M8,2,1.6218,2.1726,0.5508,0.62
188,V88,2,2.9101,3.2687,0.3586,0.93


## 15. Entity Davranış Raporu

### `entity_behavior_report(df)`
Entity bazında (kart, email domain, cihaz tipi) işlem istatistikleri hesaplar:
- `tx_count`, `amt_mean/std/median`, `amt_zscore`
- `velocity_1h`, `velocity_1d` — kısa süreli işlem yoğunluğu
- `fraud_rate` ve yüksek riskli entity tespiti (global rate × 2x eşiği)

**Kullanılan entity kolonları:** `card1`, `P_emaildomain`, `DeviceType`

> **Bulgu:** 762 yüksek riskli entity tespit edildi.  
> Bu entity'lerin `fraud_rate` = 1.0 — yani o entite'den gelen tüm işlemler fraud.

In [46]:
def entity_behavior_report(
    df: pd.DataFrame,
    entity_cols: list = None,
    amount_col: str = "TransactionAmt",
    time_col: str = "TransactionDT",
    target_col: str = "isFraud",
    velocity_windows: list = [3600, 86400],   # saniye: 1s, 1gün
    min_tx_count: int = 3,
) -> dict:
    """
    Entity bazında davranış profili çıkarır.

    Döndürdüğü sözlük:
    - 'entity_profiles'   : entity başına özet istatistikler (DataFrame)
    - 'velocity_profiles' : entity başına velocity metrikleri (DataFrame)
    - 'high_risk_entities': fraud_rate'i yüksek entity'ler (DataFrame)
    - 'meta'              : koşturulan entity sayısı, parametre özeti

    Tasarım prensipleri:
    - Sadece işaretler, silmez/değiştirmez
    - Adım 3 feature engineering ve Adım 6 trusted-entity için
      doğrudan besleme yapısında döner
    - min_tx_count: küçük örnekler gürültülüdür, filtreler ama silmez
    """
    import pandas as pd
    import numpy as np

    if entity_cols is None:
        entity_cols = ["card1", "P_emaildomain", "DeviceType"]

    results = {}
    all_profiles = []
    all_velocity = []

    for entity in entity_cols:
        if entity not in df.columns:
            continue

        grp = df.groupby(entity)

        # ── Temel istatistikler ──────────────────────────────────────
        profile = grp.agg(
            tx_count=(amount_col, "count"),
            amt_mean=(amount_col, "mean"),
            amt_std=(amount_col, "std"),
            amt_min=(amount_col, "min"),
            amt_max=(amount_col, "max"),
            amt_median=(amount_col, "median"),
        ).reset_index()

        if target_col in df.columns:
            fraud_stats = grp[target_col].agg(
                fraud_count="sum",
                fraud_rate="mean",
            ).reset_index()
            profile = profile.merge(fraud_stats, on=entity, how="left")

        profile["entity_col"] = entity
        profile.rename(columns={entity: "entity_value"}, inplace=True)
        profile = profile[profile["tx_count"] >= min_tx_count]
        all_profiles.append(profile)

        # ── Velocity (zaman bazlı yoğunluk) ──────────────────────────
        if time_col in df.columns:
            sub = df[[entity, time_col, amount_col]].copy()
            sub = sub.sort_values([entity, time_col])
            sub["_rank"] = sub.groupby(entity).cumcount()

            vel_rows = []
            for window in velocity_windows:
                label = f"tx_per_{window}s"
                # Her entity için pencere içindeki tx sayısı: rolling count
                counts = (
                    sub.groupby(entity)[time_col]
                    .apply(
                        lambda t: (
                            t.expanding()
                            .apply(
                                lambda x: ((x[-1] - x) <= window).sum(),  # <-- burayı değiştir
                                raw=True,
                            )
                            .max()
                        )
                    )
                    .reset_index()
                )
                counts.columns = [entity, label]
                vel_rows.append(counts.set_index(entity))

            if vel_rows:
                vel_df = pd.concat(vel_rows, axis=1).reset_index()
                vel_df["entity_col"] = entity
                vel_df.rename(columns={entity: "entity_value"}, inplace=True)
                all_velocity.append(vel_df)

    # ── Birleştir ────────────────────────────────────────────────────
    entity_profiles_df = (
        pd.concat(all_profiles, ignore_index=True) if all_profiles else pd.DataFrame()
    )
    velocity_df = (
        pd.concat(all_velocity, ignore_index=True) if all_velocity else pd.DataFrame()
    )

    # ── Yüksek riskli entity'ler ─────────────────────────────────────
    high_risk = pd.DataFrame()
    if "fraud_rate" in entity_profiles_df.columns:
        global_rate = df[target_col].mean() if target_col in df.columns else None
        high_risk = entity_profiles_df[
            entity_profiles_df["fraud_rate"]
            > (global_rate * 2 if global_rate else 0.1)
        ].sort_values("fraud_rate", ascending=False)

    results["entity_profiles"] = entity_profiles_df
    results["velocity_profiles"] = velocity_df
    results["high_risk_entities"] = high_risk
    results["meta"] = {
        "entity_cols_used": [e for e in entity_cols if e in df.columns],
        "velocity_windows_seconds": velocity_windows,
        "min_tx_count": min_tx_count,
        "global_fraud_rate": df[target_col].mean() if target_col in df.columns else None,
        "high_risk_threshold": "fraud_rate > 2× global_rate",
    }

    return results

In [47]:
# Önce merge edilmiş df'in hazır olduğunu varsayıyoruz
# (transaction + identity join)

entity_results = entity_behavior_report(
    df=merged_df,
    entity_cols=["card1", "P_emaildomain", "DeviceType"],
    amount_col="TransactionAmt",
    time_col="TransactionDT",
    target_col="isFraud",
    velocity_windows=[3600, 86400],  # 1 saat, 1 gün
    min_tx_count=3,
)

# Sonuçlara erişim
profiles      = entity_results["entity_profiles"]
velocity      = entity_results["velocity_profiles"]
high_risk     = entity_results["high_risk_entities"]
meta          = entity_results["meta"]

# Hızlı kontrol
print(meta)
print(profiles.head())
print(f"\nYüksek riskli entity sayısı: {len(high_risk)}")
print(high_risk.head(10))

{'entity_cols_used': ['card1', 'P_emaildomain', 'DeviceType'], 'velocity_windows_seconds': [3600, 86400], 'min_tx_count': 3, 'global_fraud_rate': 0.03499000914417313, 'high_risk_threshold': 'fraud_rate > 2× global_rate'}
  entity_value  tx_count    amt_mean    amt_std  amt_min  amt_max  amt_median  \
0         1001         3   79.666667  89.494879     27.0    183.0       29.00   
1         1004         5  136.400000  93.577775     30.0    226.0      150.00   
2         1006         3  133.333333  28.867513    100.0    150.0      150.00   
3         1008         3   60.816667  37.831545     24.5    100.0       57.95   
4         1009         5  130.000000  75.828754     50.0    200.0      150.00   

   fraud_count  fraud_rate entity_col  
0            0         0.0      card1  
1            0         0.0      card1  
2            0         0.0      card1  
3            0         0.0      card1  
4            0         0.0      card1  

Yüksek riskli entity sayısı: 762
     entity_value 

---

## Özet

Bu notebook ile Adım 1 ve 2 tamamlandı. Üretilen altyapı:

| Fonksiyon | Çıktı |
|-----------|-------|
| `merge_datasets` | 590,540 satır, 434 kolon |
| `detect_column_types` | Numeric/categorical/datetime ayrımı |
| `missing_report` / `dataframe_report` | Kolon bazlı kalite profili |
| `cardinality_report` | Yüksek/orta/düşük kardinalite etiketleri |
| `constant_columns` | Quasi-constant kolon listesi |
| `duplicate_report` | 3 tam satır tekrarı (silinmedi) |
| `validity_report` | D4, D6, D11'de negatif değer tespiti |
| `outlier_report` | IQR bazlı uç değer işaretleme |
| `distribution_report` | Skewness / log transform adayları |
| `quality_summary` | 🟢/🟡/🔴 kalite bayrakları |
| `rare_combination_report` | Yüksek fraud lift'li ikili kombinasyonlar |
| `column_relationship_report` | numeric_corr / cat_dep / target_power |
| `entity_behavior_report` | 762 yüksek riskli entity |
